In [1]:
%load_ext autoreload
%autoreload 2
import os

# I believe this environment variable should be set before importing t
os.environ["PYt_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch as t
from transformers import AutoModelForCausalLM, AutoTokenizer
import argparse
import itertools
import random
import json
import torch.multiprocessing as mp
import time
import huggingface_hub
from datasets import config
from transformers import AutoTokenizer
import demo_config
from tqdm import tqdm
import matplotlib.pyplot as plt

# Kind of janky double importing dictionary_learning.dictionary_learning, but it works
# This is leftover from when dictionary_learning was a only used as a submodule
from dictionary_learning.dictionary_learning.utils import (
    hf_dataset_to_generator,
    hf_mixed_dataset_to_generator,
    hf_sequence_packing_dataset_to_generator,
)
from dictionary_learning.dictionary_learning.pytorch_buffer import ActivationBuffer
from dictionary_learning.dictionary_learning.evaluation import evaluate
from dictionary_learning.dictionary_learning.training import trainSAE
import dictionary_learning.dictionary_learning.utils as utils


if os.path.exists('/u/eboix/moe_distillation/sae_demo'):
    os.chdir('/u/eboix/moe_distillation/sae_demo')

In [2]:
sae_dir = '._saes_EleutherAI_pythia-70m-deduped_top_k/mlp_in_3/trainer_0/'
device = t.device('cuda' if t.cuda.is_available() else 'cpu')

# load config.json from sae_dir
config_path = os.path.join(sae_dir, 'config.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        config_data = json.load(f)
else:
    assert(False), f'Config file not found at {config_path}'

model_name = config_data['trainer']['lm_name']
layer_idx = config_data['trainer']['layer']
io = config_data['buffer']['io']
dtype = t.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", torch_dtype=dtype
)

model = utils.truncate_model(model, layer_idx)
model.eval()
model = model.to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
submodule = utils.get_submodule(model, layer_idx).mlp
submodule = submodule.to(device)
print('Model loaded:', model_name)

generator = hf_dataset_to_generator("monology/pile-uncopyrighted", 
    data_files={"validation": "val.jsonl.zst"},
    split="validation",)

activation_dim = config_data["trainer"]["activation_dim"]

activation_buffer = ActivationBuffer(
    generator,
    model,
    submodule,
    n_ctxs=config_data['buffer']['n_ctxs'],
    ctx_len=config_data['buffer']['ctx_len'],
    refresh_batch_size=config_data['buffer']['refresh_batch_size'],
    out_batch_size=config_data['buffer']['out_batch_size'],
    io=io,
    d_submodule=activation_dim,
    device=device,
)

print('Initialized activation buffer')


Model parameters before truncation: 70,426,624
Model parameters after truncation: 38,366,208
Model loaded: EleutherAI/pythia-70m-deduped
Initialized activation buffer


In [3]:
# create SparseMLP class, which has 3 layers
# the first layer is the input layer, the second layer is the sparse layer, and the third layer is the output layer
# the second layer has a topk activation
# the third layer has a gelu activation function
class SparseMLP(t.nn.Module):
    def __init__(self, d_in, d_out, dict_size, d_intermediate, k):
        super(SparseMLP, self).__init__()
        self.dict_size = dict_size
        self.encoder = t.nn.Linear(d_in, dict_size)
        self.layer1 = t.nn.Linear(dict_size, d_intermediate)
        self.layer2 = t.nn.Linear(d_intermediate, d_out)
        self.k = k

    def forward(self, x):
        x = self.encoder(x)
        topk_values, _ = t.topk(x.abs(), k=self.k, dim=-1)
        threshold = topk_values[..., -1].unsqueeze(-1)
        x = t.where(x.abs() >= threshold, x, t.zeros_like(x))
        x = self.layer1(x)
        x = t.nn.functional.gelu(x)
        x = self.layer2(x)
        return x

In [4]:
# Estimate variance of y_teacher
# first estimate mean
n_estimate_iters = 4
y_mean = t.zeros(activation_dim, device=device)
for _ in tqdm(range(n_estimate_iters)):
    x = next(activation_buffer)
    x = x.to(device)
    y_teacher = submodule(x)
    y_mean += y_teacher.mean(dim=0)
y_mean /= n_estimate_iters

y_variance = 0.0
for _ in tqdm(range(n_estimate_iters)):
    x = next(activation_buffer)
    x = x.to(device)
    y_teacher = submodule(x)
    y_variance += t.mean(t.mean((y_teacher - y_mean) ** 2, dim=0))
y_variance /= n_estimate_iters
y_variance = y_variance.item()
print('Variance in y', y_variance)


100%|████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 90.80it/s]

Variance in y 0.17641803622245789


In [13]:
# Train SparseMLP student model to match the teacher model on the activation buffer
# inputs are x from activation_buffer, outputs are submodule(x)
# loss is MSE
# train with AdamW with learning rate decay and warmup
lr = 1e-3
num_train_tokens = 50_000_000
num_train_steps = num_train_tokens // config_data['buffer']['out_batch_size']
print(num_train_steps, 'training steps')
submodule.eval()

student_model = SparseMLP(
    d_in=activation_dim,
    d_out=activation_dim,
    dict_size=config_data['trainer']['dict_size'],
    d_intermediate=4*activation_dim,
    k=config_data['trainer']['k'],
)
student_model = student_model.to(device)

optimizer = t.optim.Adam(student_model.parameters(), lr=lr)
criterion = t.nn.MSELoss()
scheduler = t.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_train_steps, eta_min=0)
# Training loop
from tqdm import tqdm

student_model.train()
running_loss = 0.0
step_count = 0
pbar = tqdm(activation_buffer, total=num_train_steps)
for x in pbar:
    x = x.to(device)
    optimizer.zero_grad()
    with t.no_grad():
        y_teacher = submodule(x)
    y_student = student_model(x)
    loss = criterion(y_student, y_teacher)
    loss.backward()
    optimizer.step()
    
    # Update running loss
    running_loss += loss.item()
    step_count += 1
    
    current_lr = scheduler.get_last_lr()[0]
    frac_var_explained = 1 - (loss.item() / y_variance)
    pbar.set_postfix(loss=loss.item(), avg_loss=running_loss/step_count, lr=current_lr, frac_var_explained=frac_var_explained)

    scheduler.step()

24414 training steps


 45%|▍| 11089/24414 [11:37<13:58, 15.90it/s, avg_loss=0.0305, frac_var_explained=0.85, loss=0.0265, lr=0.000572


In [9]:
# load autoencoder from sae_file
from dictionary_learning.dictionary_learning.trainers.top_k import AutoEncoderTopK
ae_file = os.path.join(sae_dir, 'ae.pt')
autoencoder = AutoEncoderTopK.from_pretrained(ae_file)
autoencoder = autoencoder.to(device)
autoencoder.eval()
print('Autoencoder loaded from:', ae_file)

Autoencoder loaded from: ._saes_EleutherAI_pythia-70m-deduped_top_k/mlp_in_3/trainer_0/ae.pt


In [ ]:
n_estimate_iters = 4
err = 0
with t.no_grad():
    for _ in tqdm(range(n_estimate_iters)):
        x = next(activation_buffer)
        x = x.to(device)
        x_recon = autoencoder(x)
        y_recon = submodule(x_recon)
        y_teacher = submodule(x)
        err += t.mean((y_recon - y_teacher)**2)
    err /= n_estimate_iters
print('Error in reconstruction:', err.item())
print('Fraction of variance explained by autoencoder:', 1 - (err.item() / y_variance))

100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 175.41it/s]

Error in reconstruction: 0.020456086844205856
Fraction of variance explained by autoencoder: 0.8840476445480249
